# 01 · TF-IDF logistic regression

How well can we identify sarcasm from the comment alone? This baseline gives each word a TF-IDF weight and learns a logistic regression coefficient for it. The model returns a probability for label 1 (sarcastic).

The saved run uses the balanced 100,000-comment subset from preprocessing revision `5400f52` (70,000 train, 15,000 validation, 15,000 test). Run the 100k version of notebook 00 first to create the shared 70/15/15 split. Install scikit-learn in the notebook's Python environment, then run these cells in order.

In [1]:
import csv
import json
from pathlib import Path
from time import perf_counter
from math import isclose

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
data_dir = root / "data" / "processed"

def read_split(name):
    comments, labels, sample_ids = [], [], []
    with (data_dir / f"{name}.csv").open(encoding="utf-8", newline="") as source:
        for row in csv.DictReader(source):
            comments.append(row["comment"])
            labels.append(int(row["label"]))
            sample_ids.append(row["sample_id"])
    return comments, labels, sample_ids

## Train the baseline

Use individual words and cap the vocabulary at 50,000 terms to keep the sparse feature matrix manageable. Words must appear in at least two training comments. Keep common words like “not” because they can change the meaning of a comment.

Try C = 0.1, 1, and 10. Smaller C means stronger regularization. Pick the highest validation F1 for sarcasm, using the smaller C for a tie. Time TF-IDF fitting separately so we can reuse it across the three fits.

In [2]:
train_comments, train_labels, _ = read_split("train")
validation_comments, validation_labels, _ = read_split("validation")
# Learn the vocabulary on training comments and reuse it for each C.
start = perf_counter()
vectorizer = TfidfVectorizer(max_features=50_000, min_df=2)
train_features = vectorizer.fit_transform(train_comments)
tfidf_seconds = perf_counter() - start
validation_features = vectorizer.transform(validation_comments)

trials = []
best_f1 = -1
for strength in (0.1, 1.0, 10.0):
    candidate = LogisticRegression(C=strength, solver="liblinear", max_iter=200, random_state=522)
    start = perf_counter()
    candidate.fit(train_features, train_labels)
    fit_seconds = perf_counter() - start
    validation_predictions = candidate.predict_proba(validation_features)[:, 1] >= 0.5
    score = f1_score(validation_labels, validation_predictions)
    training_predictions = candidate.predict_proba(train_features)[:, 1] >= 0.5
    trial = {
        "C": strength,
        "training_accuracy": accuracy_score(train_labels, training_predictions),
        "validation_accuracy": accuracy_score(validation_labels, validation_predictions),
        "validation_f1": score,
        "classifier_fit_seconds": fit_seconds,
    }
    trials.append(trial)
    if score > best_f1:
        baseline, selected, best_f1 = candidate, trial, score
    print(trial)

print(f"Selected C = {selected['C']}. Training comments: {len(train_labels):,}")

{'C': 0.1, 'training_accuracy': 0.6913714285714285, 'validation_accuracy': 0.6737333333333333, 'validation_f1': 0.6506282124500286, 'classifier_fit_seconds': 0.07737400000000005}
{'C': 1.0, 'training_accuracy': 0.7484571428571428, 'validation_accuracy': 0.6754, 'validation_f1': 0.6646001239925604, 'classifier_fit_seconds': 0.20058170800000008}
{'C': 10.0, 'training_accuracy': 0.7931, 'validation_accuracy': 0.6625333333333333, 'validation_f1': 0.6581577525661805, 'classifier_fit_seconds': 0.4002119590000002}
Selected C = 1.0. Training comments: 70,000


## Evaluate on validation and test

Use the same fitted model and a fixed probability threshold of 0.5 for both splits. Precision, recall, and F1 below refer to the sarcastic class. Majority-class accuracy shows what we'd get by always predicting the most common training label.

In [3]:
# This gives us a simple score to beat: guess the most common training label every time.
majority_label = int(sum(train_labels) > len(train_labels) / 2)
metrics = {"training": {
    "rows": len(train_labels), "selected_C": selected["C"],
    "accuracy": selected["training_accuracy"], "tfidf_fit_seconds": tfidf_seconds,
    "training_seconds": tfidf_seconds + selected["classifier_fit_seconds"], "trials": trials,
}}
prediction_dir = data_dir / "logistic_regression_predictions"
prediction_dir.mkdir(exist_ok=True)

for split in ("validation", "test"):
    comments, labels, sample_ids = read_split(split)
    # Column 1 gives the probability of sarcasm because our labels are 0 and 1.
    probabilities = baseline.predict_proba(vectorizer.transform(comments))[:, 1]
    # Count a comment as sarcastic when its probability is at least 50%.
    predictions = (probabilities >= 0.5).astype(int)
    metrics[split] = {
        "rows": len(labels),
        "accuracy": accuracy_score(labels, predictions),
        "majority_accuracy": labels.count(majority_label) / len(labels),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
        "confusion_matrix": confusion_matrix(labels, predictions, labels=[0, 1]).tolist(),
    }
    # Recalculate the scores from the four counts to catch label or axis mixups.
    (tn, fp), (fn, tp) = metrics[split]["confusion_matrix"]
    def ratio(a, b):
        return a / b if b else 0.0
    matrix_scores = {
        "accuracy": ratio(tn + tp, len(labels)), "precision": ratio(tp, tp + fp),
        "recall": ratio(tp, tp + fn), "f1": ratio(2 * tp, 2 * tp + fp + fn),
    }
    assert tn + fp + fn + tp == len(labels)
    assert all(isclose(metrics[split][key], value) for key, value in matrix_scores.items())
    # Keep the IDs so we can line these predictions up with the team's other models.
    with (prediction_dir / f"{split}.csv").open("w", encoding="utf-8", newline="") as target:
        writer = csv.writer(target)
        writer.writerow(("sample_id", "label", "prediction", "sarcasm_probability"))
        writer.writerows(zip(sample_ids, labels, predictions, probabilities))

    print(split, json.dumps(metrics[split], indent=2))

metrics_dir = root / "results" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)
(metrics_dir / "logistic_regression.json").write_text(json.dumps(metrics, indent=2) + "\n")

validation {
  "rows": 15000,
  "accuracy": 0.6754,
  "majority_accuracy": 0.5,
  "precision": 0.6874732791791364,
  "recall": 0.6432,
  "f1": 0.6646001239925604,
  "confusion_matrix": [
    [
      5307,
      2193
    ],
    [
      2676,
      4824
    ]
  ]
}
test {
  "rows": 15000,
  "accuracy": 0.6720666666666667,
  "majority_accuracy": 0.5,
  "precision": 0.6844889206576126,
  "recall": 0.6384,
  "f1": 0.6606416005519145,
  "confusion_matrix": [
    [
      5293,
      2207
    ],
    [
      2712,
      4788
    ]
  ]
}


Read each confusion matrix by row (actual label) and column (predicted label), in the order 0, 1. The lower-left count is the number of sarcastic comments the model missed. The upper-right count is the number of non-sarcastic comments it flagged.